# Sprint 4 — RAG e assistente de troubleshooting

Pipeline completo: chunking por estrutura técnica, embeddings e índice vetorial, recuperação com re-ranking operacional, prompt fundamentado, memória curta, avaliação sobre 20 perguntas e três cenários de falha.

## Preparação e opções de backend

O modo padrão usa vetores TF-IDF e índice `NearestNeighbors`, suficiente para execução offline. A avaliação Colab versionada utilizou Sentence Transformers, FAISS e Qwen 3B em GPU T4. O assistente valida a saída do LLM, tenta revisão automática e usa fallback rastreável quando a resposta não passa nos guardrails.

In [1]:
from pathlib import Path
import sys, json

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/pln_motores_rag')]
ROOT = next((p for p in candidates if (p / 'src' / 'pln_motores').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Abra o notebook na raiz do repositório ou clone o projeto em /content/pln_motores_rag.')
sys.path.insert(0, str(ROOT / 'src'))
(ROOT / 'outputs').mkdir(exist_ok=True)
print('Raiz do projeto:', ROOT)


Raiz do projeto: C:\Users\Nicolas\projects\pln_motores_rag


In [2]:
import json
import pandas as pd
from IPython.display import display, Markdown

from pln_motores.rag import TechnicalRAG, chunk_markdown
from pln_motores.assistant import TroubleshootingAssistant, SYSTEM_PROMPT
from pln_motores.llm_backends import qwen_local, openai_compatible

qa = json.loads((ROOT / 'data' / 'perguntas_troubleshooting.json').read_text(encoding='utf-8'))
print('Perguntas de avaliação:', len(qa))

Perguntas de avaliação: 20


## 1. Chunking inteligente

Os cabeçalhos são preservados como metadados. Seções longas são divididas por parágrafos com pequena sobreposição, sem cortar arbitrariamente uma instrução.

In [3]:
documents = sorted((ROOT / 'data' / 'documentos_tecnicos').glob('*.md'))
chunks = [chunk for doc in documents for chunk in chunk_markdown(doc)]
display(pd.DataFrame([c.to_dict() for c in chunks])[
    ['chunk_id', 'document', 'heading', 'equipment_types', 'anomaly_types']
].head(10))
print('Total de chunks:', len(chunks))

,chunk_id,document,heading,equipment_types,anomaly_types
0,94fae86253,datasheet_limites_mt.md,Datasheet Operacional - Motores MT Série 040,[motor],[]
1,a201b53d37,datasheet_limites_mt.md,Limites de temperatura,[mancal],"[mecânica, térmica]"
2,852b78153e,datasheet_limites_mt.md,Limites de vibração,[motor],"[mecânica, térmica]"
3,85db31e946,datasheet_limites_mt.md,Limites elétricos,[motor],[elétrica]
4,f5066ed93d,datasheet_limites_mt.md,Sistema de refrigeração,"[motor, ventilador]","[elétrica, mecânica, térmica]"
5,b0085ea21a,datasheet_limites_mt.md,Dados nominais simulados,"[motor, mancal]","[elétrica, mecânica]"
6,daf7a98476,ficha_manutencao_mt.md,Ficha de Manutenção - Motores MT Série 040,[motor],[]
7,dcebe34cb2,ficha_manutencao_mt.md,Procedimento de inspeção do sistema de refrige...,[ventilador],"[elétrica, térmica]"
8,23e30410a4,ficha_manutencao_mt.md,Procedimento de reaperto elétrico,[motor],"[elétrica, térmica]"
9,96569fafe6,ficha_manutencao_mt.md,Procedimento de verificação de alinhamento,[motor],"[mecânica, térmica]"


Total de chunks: 24


## 2. Embeddings, indexação e retriever

O score final combina similaridade vetorial do texto, similaridade do cabeçalho, cobertura lexical e bônus de metadados do estado atual. Isso permite priorizar, por exemplo, segurança e parada para um alerta crítico.

In [4]:
USE_SEMANTIC_EMBEDDINGS = False  # True no Colab com internet/GPU
embedding_model = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' if USE_SEMANTIC_EMBEDDINGS else None
rag = TechnicalRAG(embedding_model=embedding_model).build(ROOT / 'data' / 'documentos_tecnicos')
print('Backend:', rag.backend)

sample = rag.retrieve(
    'A vibração subiu com ruído metálico. O que fazer?',
    {'tipo_equipamento': 'motor', 'tipo_anomalia': 'mecânica', 'severidade': 'critico'},
    top_k=4,
)
display(pd.DataFrame(sample)[['document', 'heading', 'base_score', 'heading_score', 'score']])

Backend: tfidf+sklearn-nn


,document,heading,base_score,heading_score,score
0,datasheet_limites_mt.md,Limites de vibração,0.153896,0.115260,0.363526
1,manual_motor_mt.md,Vibração elevada no mancal,0.140662,0.079578,0.340704
2,manual_motor_mt.md,"Segurança, bloqueio e desenergização",0.108211,0.000000,0.333606
3,manual_motor_mt.md,Inspeção preventiva semanal,0.092209,0.000000,0.258104


### Qualidade da recuperação

O gabarito associa cada pergunta a uma ou mais seções relevantes. Reportamos Precision@1, Precision@4, Hit@4 e MRR.

In [5]:
retrieval_metrics = rag.evaluate_retrieval(qa, top_k=4)
display(pd.Series({k: v for k, v in retrieval_metrics.items() if k != 'details'}, name='valor').to_frame())

,valor
precision_at_k,0.287500
precision_at_1,0.850000
hit_at_k,1.000000
mrr,0.916667


## 3. Assistente conversacional

O prompt exige persona técnica, fundamentação, citação, confiança, abstenção e segurança. A memória guarda somente os quatro turnos mais recentes. O guardrail rejeita citações inválidas, baixa sustentação, abstenção indevida, procedimentos temporais incorretos e contradições de parada segura.

In [6]:
print(SYSTEM_PROMPT)

LLM_BACKEND = 'extractive'  # 'extractive', 'qwen' ou 'api'
if LLM_BACKEND == 'qwen':
    # Requer: %pip install transformers accelerate torch
    llm = qwen_local(
        'Qwen/Qwen2.5-3B-Instruct',
        max_new_tokens=320,
        system_prompt=SYSTEM_PROMPT,
    )
elif LLM_BACKEND == 'api':
    # Defina OPENAI_API_KEY e ajuste o modelo/base_url conforme o provedor.
    llm = openai_compatible(model='SEU_MODELO', base_url=None)
else:
    llm = None

assistant = TroubleshootingAssistant(rag, llm=llm, memory_turns=4)

Você é um assistente técnico especialista em motores elétricos.
Responda em português técnico claro, somente com base no contexto documental e operacional fornecido.
Não invente estado, ação já executada, dimensão do motor, valor, limite, causa ou diagnóstico.
Não trate severidade como nível de confiança. A confiança depende apenas da suficiência das evidências.
Priorize segurança: não instrua intervenção energizada e não substitua procedimentos locais.
Responda diretamente à pergunta, distinguindo verificações recomendadas de critérios de parada.
Cite somente fontes efetivamente usadas, no formato exato [arquivo > seção].
Se faltar evidência, declare explicitamente que não há informação suficiente.
Finalize exatamente com: Confiança: alta, média ou baixa — justificativa breve baseada nas evidências.


## 4. Três cenários demonstrativos

In [7]:
scenarios = [
    ('Anomalia elétrica', 'A corrente das fases está desequilibrada. O que verifico e quando devo parar?', {
        'tipo_equipamento': 'motor', 'equipamento_id': 'Motor MT-042', 'tipo_anomalia': 'elétrica',
        'sensor_tipo': 'corrente_fase', 'severidade': 'moderado',
        'resumo_alerta': 'Alerta moderado: corrente 12% acima do equilíbrio esperado no sensor CUR-042-1.'}),
    ('Anomalia mecânica', 'A vibração do mancal subiu e surgiu ruído metálico. Qual é a ação segura?', {
        'tipo_equipamento': 'motor', 'equipamento_id': 'Motor MT-042', 'tipo_anomalia': 'mecânica',
        'sensor_tipo': 'vibracao_mancal', 'severidade': 'critico',
        'resumo_alerta': 'Alerta crítico: vibração 8,1 mm/s RMS no sensor VIB-042-1.'}),
    ('Manutenção preventiva', 'Quais verificações devo executar na manutenção trimestral?', {
        'tipo_equipamento': 'motor', 'equipamento_id': 'Motor MT-042', 'tipo_anomalia': 'preventiva',
        'severidade': 'leve', 'resumo_alerta': 'Tendência leve de temperatura nas duas últimas rondas.'}),
]

for name, question, context in scenarios:
    assistant.history.clear()
    result = assistant.ask(question, context, top_k=3)
    display(Markdown(
        f'### {name}\n\n**Pergunta:** {question}\n\n**Resposta:** {result["answer"]}'
        f'\n\n**Modo de geração:** `{result["generation_mode"]}`'
    ))

### Anomalia elétrica

**Pergunta:** A corrente das fases está desequilibrada. O que verifico e quando devo parar?

**Resposta:** Situação atual: Alerta moderado: corrente 12% acima do equilíbrio esperado no sensor CUR-042-1.

Verificações recomendadas:
- Desequilíbrio de corrente entre fases Correntes de fase desequilibradas podem indicar tensão desequilibrada, conexão frouxa, contato degradado, problema no enrolamento ou carga anormal. [manual_motor_mt.md > Desequilíbrio de corrente entre fases]
- Registre corrente e tensão nas três fases sob condição estável. [manual_motor_mt.md > Desequilíbrio de corrente entre fases]
- Entre 5% e 10%, verificar carga, tensão e conexões. [datasheet_limites_mt.md > Limites elétricos]

Critério de parada:
- Acima de 10%, reduzir a carga quando seguro e programar inspeção elétrica imediata. [datasheet_limites_mt.md > Limites elétricos]

Confiança: alta — resposta limitada às evidências recuperadas e ao contexto operacional.

**Modo de geração:** `extractive`

### Anomalia mecânica

**Pergunta:** A vibração do mancal subiu e surgiu ruído metálico. Qual é a ação segura?

**Resposta:** Situação atual: Alerta crítico: vibração 8,1 mm/s RMS no sensor VIB-042-1.

Verificações recomendadas:
- Inspecione fixações, base, acoplamento e condição do mancal. [manual_motor_mt.md > Vibração elevada no mancal]
- Após qualquer correção, confirme a redução da vibração em condição operacional equivalente. [manual_motor_mt.md > Vibração elevada no mancal]
- Procedimento de substituição do mancal Desenergize e bloqueie o conjunto, desacople a carga e marque a posição dos componentes. [ficha_manutencao_mt.md > Procedimento de substituição do mancal]

Critério de parada:
- Um crescimento rápido, ruído metálico ou temperatura simultaneamente alta requer parada segura. [manual_motor_mt.md > Vibração elevada no mancal]

Confiança: alta — resposta limitada às evidências recuperadas e ao contexto operacional.

**Modo de geração:** `extractive`

### Manutenção preventiva

**Pergunta:** Quais verificações devo executar na manutenção trimestral?

**Resposta:** Situação atual: Tendência leve de temperatura nas duas últimas rondas.

Verificações recomendadas:
- Manutenção preventiva trimestral A rotina trimestral inclui limpeza do sistema de ventilação, verificação do torque das conexões conforme tabela aplicável, inspeção do acoplamento, avaliação de alinhamento, conferência da base e revisão das tendências de corrente, temperatura e vibração. [manual_motor_mt.md > Manutenção preventiva trimestral]
- Com o motor desenergizado, inspecione a caixa de ligação e meça resistência de isolamento quando previsto no plano. [manual_motor_mt.md > Manutenção preventiva trimestral]
- Ficha de Manutenção - Motores MT Série 040 Documento sintético para demonstração do fluxo de troubleshooting. [ficha_manutencao_mt.md > Ficha de Manutenção - Motores MT Série 040]

Confiança: alta — resposta limitada às evidências recuperadas e ao contexto operacional.

**Modo de geração:** `extractive`

## 5. Avaliação conversacional

As métricas locais são auditáveis: faithfulness verifica suporte sentencial no documento/contexto operacional; answer relevancy combina cobertura do gabarito e cosseno TF-IDF; context precision compara as seções recuperadas ao gabarito. RAGAS ou avaliação por especialistas deve complementar esses proxies.

In [8]:
assistant_metrics = assistant.evaluate(qa, top_k=3)
display(pd.Series({k: v for k, v in assistant_metrics.items() if k not in ['details', 'method_note']}, name='valor').to_frame())
print(assistant_metrics['method_note'])

,valor
faithfulness,0.701429
answer_relevancy,0.529468
context_precision,0.383333


Faithfulness mede suporte sentencial no documento + contexto operacional; answer relevancy combina cobertura lexical do gabarito e cosseno TF-IDF; use RAGAS/LLM-judge como validação complementar.


### Execução semântica com Qwen 3B e guardrails

Os artefatos abaixo foram gerados em Google Colab com GPU T4, Sentence Transformers + FAISS, `Qwen/Qwen2.5-3B-Instruct`, `top_k=3` e validação pós-geração. Eles preservam as 20 respostas, os três cenários e a demonstração de memória.

In [9]:
qwen_metrics = json.loads((ROOT / 'outputs' / 'metricas_qwen_hibrido.json').read_text(encoding='utf-8'))
qwen_evaluation = json.loads((ROOT / 'outputs' / 'avaliacao_qwen_hibrido.json').read_text(encoding='utf-8'))
qwen_demo = json.loads((ROOT / 'outputs' / 'demonstracao_qwen_hibrido.json').read_text(encoding='utf-8'))

display(pd.Series({
    'Precision@1': qwen_metrics['precision_at_1'],
    'Hit@3': qwen_metrics['hit_at_3'],
    'MRR': qwen_metrics['mrr'],
    'Context precision@3': qwen_metrics['context_precision'],
    'Faithfulness': qwen_metrics['faithfulness'],
    'Answer relevancy': qwen_metrics['answer_relevancy'],
}, name='valor').to_frame())
display(pd.Series(qwen_metrics['generation_modes'], name='respostas').to_frame())
print('Memória injetada:', qwen_demo['demonstracao_memoria']['memoria_anterior_injetada'])
print('Turnos armazenados:', qwen_demo['demonstracao_memoria']['turnos_armazenados'])

display(pd.DataFrame(qwen_evaluation)[
    ['id', 'generation_mode', 'faithfulness', 'answer_relevancy', 'context_precision']
])

for item in qwen_demo['cenarios']:
    display(Markdown(
        f'#### {item["cenario"].capitalize()} — execução Qwen protegida'
        f'\n\n**Contexto:** {item["contexto"]["resumo_alerta"]}'
        f'\n\n**Resposta final:** {item["answer"]}'
        f'\n\n**Modo:** `{item["generation_mode"]}`'
    ))

memory = qwen_demo['demonstracao_memoria']
display(Markdown(
    f'#### Continuação com memória'
    f'\n\n**Pergunta:** {memory["pergunta_continuacao"]}'
    f'\n\n**Resposta:** {memory["answer"]}'
))

,valor
Precision@1,0.900000
Hit@3,1.000000
MRR,0.950000
Context precision@3,0.400000
Faithfulness,0.723864
Answer relevancy,0.572087


,respostas
guardrail_fallback,14
llm_revised,3
llm,3


Memória injetada: True
Turnos armazenados: 2


,id,generation_mode,faithfulness,answer_relevancy,context_precision
0,TS-01,guardrail_fallback,0.600000,0.448193,0.333333
1,TS-02,llm_revised,1.000000,0.788013,0.333333
2,TS-03,llm_revised,0.800000,0.758260,0.333333
3,TS-04,guardrail_fallback,0.800000,0.521934,0.666667
4,TS-05,guardrail_fallback,0.600000,0.464357,0.666667
5,TS-06,llm_revised,0.800000,0.451078,0.333333
6,TS-07,guardrail_fallback,0.600000,0.779679,0.333333
7,TS-08,guardrail_fallback,0.600000,0.710687,0.666667
8,TS-09,guardrail_fallback,0.800000,0.768339,0.333333
9,TS-10,guardrail_fallback,0.800000,0.412160,0.333333


#### Anomalia elétrica — execução Qwen protegida

**Contexto:** Alerta moderado detectado no Motor MT-042. A corrente apresentou desequilíbrio de 12% no sensor CUR-042-1.

**Resposta final:** Situação atual: Alerta moderado detectado no Motor MT-042. A corrente apresentou desequilíbrio de 12% no sensor CUR-042-1.

Verificações recomendadas:
- Desequilíbrio de corrente entre fases Correntes de fase desequilibradas podem indicar tensão desequilibrada, conexão frouxa, contato degradado, problema no enrolamento ou carga anormal. [manual_motor_mt.md > Desequilíbrio de corrente entre fases]
- Entre 5% e 10%, verificar carga, tensão e conexões. [datasheet_limites_mt.md > Limites elétricos]
- Depois do reaperto, registre os valores de resistência entre conexões quando aplicável e acompanhe corrente e temperatura nas três fases durante o retorno controlado. [ficha_manutencao_mt.md > Procedimento de reaperto elétrico]

Critério de parada:
- Acima de 10%, reduzir a carga quando seguro e programar inspeção elétrica imediata. [datasheet_limites_mt.md > Limites elétricos]

Confiança: alta — resposta limitada às evidências recuperadas e ao contexto operacional.

**Modo:** `guardrail_fallback`

#### Anomalia mecânica — execução Qwen protegida

**Contexto:** Alerta crítico no Motor MT-042: vibração de 8,1 mm/s RMS no sensor VIB-042-1, acompanhada de ruído metálico.

**Resposta final:** Situação atual: Alerta crítico no Motor MT-042: vibração de 8,1 mm/s RMS no sensor VIB-042-1, acompanhada de ruído metálico.

Verificações recomendadas:
- Vibração elevada no mancal Vibração elevada pode ser causada por desbalanceamento, desalinhamento, folga estrutural, ressonância ou degradação do mancal. [manual_motor_mt.md > Vibração elevada no mancal]
- Inspecione fixações, base, acoplamento e condição do mancal. [manual_motor_mt.md > Vibração elevada no mancal]
- Falha de mancal Indícios de falha de mancal incluem ruído repetitivo, aumento de vibração em frequências características, elevação de temperatura e presença de partículas no lubrificante. [manual_motor_mt.md > Falha de mancal]

Critério de parada:
- Um crescimento rápido, ruído metálico ou temperatura simultaneamente alta requer parada segura. [manual_motor_mt.md > Vibração elevada no mancal]

Confiança: alta — resposta limitada às evidências recuperadas e ao contexto operacional.

**Modo:** `guardrail_fallback`

#### Manutenção preventiva — execução Qwen protegida

**Contexto:** Sem alerta crítico. Foi observada tendência leve de temperatura nas duas últimas rondas.

**Resposta final:** Situação atual: Sem alerta crítico. Foi observada tendência leve de temperatura nas duas últimas rondas.

Verificações recomendadas:
- Manutenção preventiva trimestral A rotina trimestral inclui limpeza do sistema de ventilação, verificação do torque das conexões conforme tabela aplicável, inspeção do acoplamento, avaliação de alinhamento, conferência da base e revisão das tendências de corrente, temperatura e vibração. [manual_motor_mt.md > Manutenção preventiva trimestral]
- Inspeção preventiva semanal Na inspeção semanal, registre ruído, temperatura dos mancais, temperatura do enrolamento, vibração global, corrente por fase, limpeza das entradas de ar e condição aparente de cabos e fixações. [manual_motor_mt.md > Inspeção preventiva semanal]
- Registro e rastreabilidade da manutenção Cada ordem deve registrar ativo, sintoma, diagnóstico, ação executada, componente e lote quando aplicável, medições antes e depois, responsáveis e horário. [ficha_manutencao_mt.md > Registro e rastreabilidade da manutenção]

Confiança: alta — resposta limitada às evidências recuperadas e ao contexto operacional.

**Modo:** `guardrail_fallback`

#### Continuação com memória

**Pergunta:** E o que devo registrar ao concluir essa manutenção?

**Resposta:** Situação atual: Sem alerta crítico. Foi observada tendência leve de temperatura nas duas últimas rondas.

Verificações recomendadas:
- Registro e rastreabilidade da manutenção Cada ordem deve registrar ativo, sintoma, diagnóstico, ação executada, componente e lote quando aplicável, medições antes e depois, responsáveis e horário. [ficha_manutencao_mt.md > Registro e rastreabilidade da manutenção]
- Manutenção preventiva trimestral A rotina trimestral inclui limpeza do sistema de ventilação, verificação do torque das conexões conforme tabela aplicável, inspeção do acoplamento, avaliação de alinhamento, conferência da base e revisão das tendências de corrente, temperatura e vibração. [manual_motor_mt.md > Manutenção preventiva trimestral]
- Com o motor desenergizado, inspecione a caixa de ligação e meça resistência de isolamento quando previsto no plano. [manual_motor_mt.md > Manutenção preventiva trimestral]

Confiança: alta — resposta limitada às evidências recuperadas e ao contexto operacional.

## 6. Limites e teste de abstenção

O corpus é sintético e restrito. O sistema não autoriza intervenção, não inventa valores ausentes e deve declarar insuficiência quando a evidência é fraca.

In [10]:
assistant.history.clear()
outside = assistant.ask('Qual é o torque exato do terminal de um motor de outra marca, modelo ZX-900?', {'tipo_equipamento': 'motor'})
display(Markdown(outside['answer']))

Não há informação suficiente nos documentos recuperados para responder com segurança sobre ZX-900. Confiança: baixa — o identificador consultado não consta nas evidências disponíveis.